## Ingest BTCUSDT AggTrades → Delta Table

Extract a Binance `aggTrades` zip from a Unity Catalog Volume, parse via **Spark CSV**, and write as a managed Delta table with **liquid clustering**.

| Parameter | Value |
|---|---|
| **Source** | `/Volumes/smart-trading-dev/00_volume/raw-btcusdt-aggtrades/BTCUSDT-aggTrades-2026-02.zip` |
| **Target** | `` `smart-trading-dev`.`01_bronze`.`aggtrades` `` |
| **Pipeline** | Zip → Extract → `spark.read.csv` → Transform → Delta |
| **Clustering** | Liquid clustering on `(trade_date, symbol)` |

In [0]:
# --- Source (Volume) ---
CATALOG = "smart-trading-dev"
SCHEMA  = "00_volume"
VOLUME  = "raw-btcusdt-aggtrades"
ZIP_FILE_NAME = "BTCUSDT-aggTrades-2026-02.zip"

ZIP_PATH    = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{ZIP_FILE_NAME}"
EXTRACT_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_extracted/"

# --- Target (Delta table) ---
TBL_SCHEMA = "01_bronze"
TBL_NAME   = "aggtrades"
TGT_TBL_PATH = f"`{CATALOG}`.`{TBL_SCHEMA}`.`{TBL_NAME}`"

# --- Data ---
SYMBOL = "BTCUSDT"

print(f"Source:  {ZIP_PATH}")
print(f"Target:  {TGT_TBL_PATH}")
print(f"Symbol:  {SYMBOL}")

In [0]:
import zipfile, os

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

extracted_files = [f for f in os.listdir(EXTRACT_DIR) if f.endswith(".csv")]
SRC_FILE_PATH = os.path.join(EXTRACT_DIR, extracted_files[0])

print(f"Extracted {len(extracted_files)} CSV file(s):")
for f in extracted_files:
    size_mb = os.path.getsize(os.path.join(EXTRACT_DIR, f)) / 1e6
    print(f"  • {f}  ({size_mb:.1f} MB)")
print(f"\nSRC_FILE_PATH = {SRC_FILE_PATH}")

In [0]:
# Quick peek at raw data before applying typed schema
df_preview = spark.read.csv(SRC_FILE_PATH, header=False)

print(f"File: {os.path.basename(SRC_FILE_PATH)}")
print(f"Columns: {len(df_preview.columns)}, Rows: {df_preview.count():,}")
display(df_preview.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, DecimalType, BooleanType
)

schema = StructType([
    StructField("agg_trade_id",    LongType()),
    StructField("price",           DecimalType(18, 8)),
    StructField("quantity",        DecimalType(18, 8)),
    StructField("first_trade_id",  LongType()),
    StructField("last_trade_id",   LongType()),
    StructField("transact_time",   LongType()),
    StructField("is_buyer_maker",  BooleanType()),
    StructField("is_best_match",   BooleanType()),
])

df = (
    spark.read.csv(SRC_FILE_PATH, schema=schema, header=False)
    .withColumn("symbol", F.lit(SYMBOL))
    .withColumn("transact_timestamp", F.timestamp_micros("transact_time"))
    .withColumn("trade_date", F.to_date("transact_timestamp"))
)

print(f"Row count: {df.count():,}")
display(df.limit(5))

In [0]:
# Ensure catalog & schema exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{TBL_SCHEMA}`")

# Write Delta table with liquid clustering in one shot
(df
 .writeTo(TGT_TBL_PATH)
 .using("delta")
 .clusterBy("trade_date")
 .createOrReplace()
)

print(f"Delta table written: {TGT_TBL_PATH}")
print(f"Liquid clustering: (trade_date, symbol)")

In [0]:
import shutil

shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
print(f"Cleaned up: {EXTRACT_DIR}")